# 11 · Student sweep and ablation runs (GPU)

Two jobs on one T4, on the **official split** (`data.source: ids`: raw train restricted to
`Data-Pipeline/data/splits/{train,val}_ids.txt`, 132,388 / 21,363 rows). The gate set is never read.

* **`MODE = "sweep"`**: one worker's half of the Optuna sweep (`src/student/sweep.py`,
  `docs/tuning.md`): DistilBERT, mixed loss on the stack's soft labels, TPE over alpha, T,
  learning rate, epochs and max length. Shivam runs `WORKER = 0`, Nishaant `WORKER = 1`, with the
  same `STUDY`. Each trial is about 15 to 65 minutes; 15 trials fit one 12-hour session.
  `WARM_START` starts this half from the other half's finished trials (no new random trials);
  re-running a worker after a dead session: raise `SESSION`, or the random trials repeat.
* **`MODE = "ablation"`**: the ablation grid with the tuned values (`TUNED`): gold-only,
  stack only (alpha 0), mixed, and the two attribution arms (TF-IDF, calibrated Qwen), plus the
  architectures on the mixed recipe. Pick the runs in `ABLATION`.

Outputs in `/kaggle/working/` (ids, scores and metrics only, no comment text): `student/<run>/`
(model, `meta.json`, `thresholds.json`, validation prediction file, metrics; in ablation mode
also `package/`, the §2.5 model package with `model.onnx`), `student/optuna.db` (this worker's
study, when `STORAGE` is None), `mlruns/` (MLflow), `reports/` (`sweep.json`, `train.json`).

**Before the first run:**
1. Accept the rules at https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/rules,
   then **Add Input → Competitions → `jigsaw-toxic-comment-classification-challenge`**.
2. **Add-ons → Secrets**: `GCS_TEACHER_KEY` (a service-account key, the whole JSON, that can read
   `gs://distillate-mlops-data/`), or `HF_TOKEN` (read access to `distillate-team`) with
   `EXTRA = ["storage.backend=hf"]` for the Hugging Face copy (same files, same checks).
   Optional: `OPTUNA_STORAGE` (a shared study database URL), `MLFLOW_TRACKING_URI` with
   `MLFLOW_TRACKING_USERNAME` and `MLFLOW_TRACKING_PASSWORD` (the team's MLflow server; without
   the login every run gets a 401). Gold-only runs need no secret.

3. Ablation mode also scores each arm on the gate, bias and h2 sets: accept the rules of
   `jigsaw-unintended-bias-in-toxicity-classification` and add it as an input too.

**Session settings:** Accelerator **GPU T4** (one is used), Internet **On**.
Run with **"Save Version → Save & Run All"** so the outputs are kept (a sweep runs for hours).

In [ ]:
# Code version. Pin a commit hash for runs you report.
REPO_URL = "https://github.com/Nishaant-Soni/Distillate.git"
REPO_REF = "main"

MODE = "sweep"                    # "sweep" or "ablation"

# Sweep: one worker's half of the study.
STUDY = "distilbert-mixed-v1"
WORKER = 0                        # Shivam 0, Nishaant 1
SESSION = 0                       # raise by 1 to re-run a worker (new random draws)
N_TRIALS = 15                     # this worker's trials
TIMEOUT_H = 11                    # session budget: no trial starts that, at the length of the
                                  # longest trial so far, would end later (Kaggle stops at 12 h)
STORAGE = None                    # None: student/optuna.db here (merge the halves afterwards);
                                  # or the OPTUNA_STORAGE secret for one shared database
WARM_START = None                 # the other half's study file, to start from its trials: a path
                                  # in an attached Kaggle dataset, e.g.
                                  # "/kaggle/input/<dataset>/optuna/worker1.db" (the study files
                                  # are on GCS only, so a store object works with the GCS key,
                                  # not with storage.backend=hf)

# Ablation: the tuned values (from `python -m src.student.sweep best`), and the runs to train.
# Trial 26 of the merged study (28 trials): val mean AUC 0.99364 (docs/tuning.md).
TUNED = ["loss.alpha=0.3432987447660936", "loss.temperature=3.71295432376268",
         "train.lr=9.22542696771891e-05", "train.epochs=4", "model.max_length=256"]
GRID = {
    "distilbert-gold":    ["loss.alpha=1.0"],
    "distilbert-stack":   ["loss.alpha=0.0"],
    "distilbert-mixed":   [],
    "distilbert-tfidf":   ["soft_labels.arm=tfidf"],
    "distilbert-calqwen": ["soft_labels.arm=calqwen"],
    "minilm-mixed":       ["model.name=minilm"],
    "modernbert-mixed":   ["model.name=modernbert"],
    "distilbert-mixed-5ep": ["train.epochs=5"],   # the check outside the study (tuning.md)
}
ABLATION = ["distilbert-gold", "distilbert-stack", "distilbert-mixed"]   # what this session trains
# Each ablation arm is scored right after training on the gate, bias and h2 sets: the
# prediction files the gate and the bias analysis read, next to val.parquet, with
# thresholds.json. Needs competition jigsaw-unintended-bias-in-toxicity-classification too.
NAMES = {"gate": "gate", "bias": "bias_eval", "h2": "h2_eval"}   # configs/gate.yaml stems (PR #33)
PACKAGE = True                    # build each ablation run's §2.5 package (model.onnx, ...)
EXTRA = []                        # e.g. ["storage.backend=hf"] (the Hugging Face copy)

In [ ]:
import subprocess
import sys
from pathlib import Path

# iterative-stratification, mlflow, optuna, google-cloud-storage, and the ONNX export packages;
# torch and transformers come with the Kaggle image.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification",
                "mlflow", "optuna==5.0.0", "google-cloud-storage", "onnx", "onnxscript",
                "onnxruntime"], check=True)

REPO_DIR = Path("/tmp/Distillate")   # outside /kaggle/working, so the repo is not saved as output
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet", "--ff-only"], check=False)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR / "Model-Pipeline"))
print("code_commit:", CODE_COMMIT)

In [ ]:
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "no GPU: set the accelerator to GPU T4"

In [ ]:
# Secrets, into the environment. Never printed. Whichever exist are loaded.
import os

from kaggle_secrets import UserSecretsClient

for name in ("GCS_TEACHER_KEY", "HF_TOKEN", "OPTUNA_STORAGE", "MLFLOW_TRACKING_URI",
             "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):   # the server's login
    try:
        os.environ[name] = UserSecretsClient().get_secret(name)
    except Exception:   # secret not attached
        pass
if STORAGE is None and os.environ.get("OPTUNA_STORAGE"):
    STORAGE = os.environ["OPTUNA_STORAGE"]
print("store key:", "GCS_TEACHER_KEY" in os.environ, "| shared study:", STORAGE is not None,
      "| MLflow server:", "MLFLOW_TRACKING_URI" in os.environ)

## 1. Raw train (cleaned, in /tmp)

In [ ]:
from src.teacher.config import load_config as load_teacher_config
from src.teacher.data import check_frame, find_competition_dir, read_competition_split, to_raw_frame

DATA = Path("/tmp/wiki")
DATA.mkdir(parents=True, exist_ok=True)
RAW_TRAIN = DATA / "wiki_train_raw.parquet"

teacher_cfg = load_teacher_config()
comp_dir = find_competition_dir("/kaggle/input")   # fails if the competition is not attached
raw = to_raw_frame(read_competition_split(comp_dir, "train"))
print(check_frame(raw, teacher_cfg["data"]["expected_rows"]["train"]))   # 159,571 rows
raw.to_parquet(RAW_TRAIN, index=False)
del raw

In [ ]:
# Paths and data for every run: the official split by id; outputs under /kaggle/working.
OUT = Path("/kaggle/working")
BASE = [
    f"paths.raw_train={RAW_TRAIN}",
    f"paths.split_ids_dir={REPO_DIR / 'Data-Pipeline' / 'data' / 'splits'}",
    f"paths.out_dir={OUT / 'student'}",
    f"paths.mlflow_db={OUT / 'mlruns' / 'mlflow.db'}",
    f"paths.mlflow_artifacts={OUT / 'mlruns' / 'artifacts'}",
    f"paths.reports_dir={OUT / 'reports'}",
    "paths.cache_dir=/tmp/cache",
    "data.source=ids",
    *EXTRA,
]

## 2. Sweep (`MODE = "sweep"`)

In [ ]:
import json
import shutil

from src.student import sweep
from src.student.config import load_config


def warm_start_file(where):
    """A writable local copy of the other half's study file (a path, or a store object)."""
    local = Path("/tmp/warm_start.db")
    if Path(where).exists():
        shutil.copy(where, local)
    else:
        from src.teacher.storage import get_storage

        base = load_config(None, BASE)
        got = get_storage(base, base["soft_labels"].get("local_root")).download(
            where, Path("/tmp/warm"))
        shutil.copy(got, local)
    return f"sqlite:///{local}"


if MODE == "sweep":
    sets = BASE + [
        "run_name=distilbert-mixed", "loss.alpha=0.5", "mlflow.experiment=distillate-student-sweep",
        f"sweep.study_name={STUDY}", f"sweep.worker={WORKER}", f"sweep.session={SESSION}",
        f"sweep.n_trials={N_TRIALS}", f"sweep.timeout_h={TIMEOUT_H}",
    ] + ([f"sweep.storage={STORAGE}"] if STORAGE else []) \
      + ([f"sweep.warm_start={warm_start_file(WARM_START)}"] if WARM_START else [])
    result = sweep.run(load_config(None, sets))
    print(json.dumps({k: result[k] for k in ("study_name", "storage", "trials", "states",
                                             "warm_start_copied")}, indent=2))
    print("best:", json.dumps(result["best"], indent=2, default=str))

## 3. Ablation grid (`MODE = "ablation"`)

In [ ]:
import glob

import pandas as pd

from src.teacher.data import (clean_text, find_competition_dir, read_competition_split,
                              to_raw_frame)

SPLITS = REPO_DIR / "Data-Pipeline" / "data" / "splits"
SCORE_BATCH = 256


def ids(name):
    return [x.strip() for x in (SPLITS / f"{name}_ids.txt").read_text().splitlines() if x.strip()]


def take(frame, wanted, name):
    frame = frame.drop_duplicates("id").set_index("id")
    missing = [i for i in wanted if i not in frame.index]
    if missing:
        raise ValueError(f"{name}: {len(missing)} ids not in the source, e.g. {missing[:3]}")
    return frame.loc[wanted, ["text"]].reset_index()


def eval_texts():
    """The gate, bias and h2 texts by id (id lists in git), cleaned as in training."""
    wiki = to_raw_frame(read_competition_split(find_competition_dir("/kaggle/input"), "test"))
    out = {"gate": take(wiki[["id", "text"]], ids("gate"), "gate")}
    found = sorted(glob.glob("/kaggle/input/**/test_private_expanded.csv*", recursive=True))
    assert len(found) == 1, f"attach the unintended-bias competition, found {found}"
    civil = pd.read_csv(found[0], dtype=str, keep_default_na=False, usecols=["id", "comment_text"])
    civil = pd.DataFrame({"id": civil["id"].astype(str),
                          "text": civil["comment_text"].map(clean_text)})
    out["bias"] = take(civil, ids("bias_eval"), "bias")
    out["h2"] = take(civil, ids("h2_eval"), "h2")
    return out


if MODE == "ablation":   # loaded before the first arm trains, so a missing input fails early
    TEXTS = eval_texts()
    print({k: len(v) for k, v in TEXTS.items()})   # gate 63,978, bias 21,577, h2 10,000

import json
import shutil
import time

import numpy as np
import torch

from src.student import train
from src.student.model import load_trained
from src.student.predict import check_prediction_frame, predict_logits, prediction_frame


def score(run_dir, device=None):
    """The gate, bias and h2 prediction files for one trained arm, next to its val.parquet."""
    run_dir = Path(run_dir)
    device = device or torch.device("cuda")
    meta = json.loads((run_dir / "model" / "meta.json").read_text())
    model, tokenizer = load_trained(run_dir / "model")
    model.to(device)
    rows = {}
    for key, frame in TEXTS.items():
        start = time.perf_counter()
        texts = frame["text"].tolist()
        order = np.argsort([len(x) for x in texts], kind="stable")   # less padding
        logits = predict_logits(model, tokenizer, [texts[i] for i in order],
                                int(meta["max_length"]), SCORE_BATCH, device, fp16=True)
        unsorted = np.empty_like(logits)
        unsorted[order] = logits
        pred = prediction_frame(frame["id"], unsorted, meta["model_version"])
        check_prediction_frame(pred)
        pred.to_parquet(run_dir / "predictions" / f"{NAMES[key]}.parquet", index=False)
        rows[NAMES[key]] = {"rows": len(pred), "seconds": round(time.perf_counter() - start, 1)}
    shutil.copy(run_dir / "model" / "thresholds.json", run_dir / "predictions" / "thresholds.json")
    (run_dir / "scoring.json").write_text(json.dumps(
        {"code_commit": CODE_COMMIT, "model_version": meta["model_version"], "files": rows},
        indent=2) + "\n")
    del model
    torch.cuda.empty_cache()
    return rows


runs, failed = {}, {}
if MODE == "ablation":
    for name in ABLATION:
        sets = BASE + TUNED + [f"run_name={name}", "mlflow.experiment=distillate-student-ablation",
                               f"package.enabled={str(PACKAGE).lower()}", *GRID[name]]
        try:     # one failed arm (out of memory, a download) doesn't lose the others
            runs[name] = train.run(load_config(None, sets))
            print(name, round(runs[name]["best_val_mean_auc"], 4),
                  "| package:", runs[name].get("package", {}).get("status"))
            m = json.loads((Path(runs[name]["run_dir"]) / "metrics.json").read_text())
            print("  val PR-AUC per label:", json.dumps({k: m[k]["pr_auc"] for k in m}))
            print("  scored:", score(runs[name]["run_dir"]))
        except Exception as e:
            failed[name] = f"{type(e).__name__}: {e}"
            print(name, "FAILED:", failed[name])
            torch.cuda.empty_cache()
    print("failed:", failed or "none")


## 4. Results (official validation split, unweighted)

In [ ]:
import pandas as pd

from src.teacher.config import LABELS

if MODE == "sweep":
    table = pd.DataFrame(result["trials_table"]).sort_values("value", ascending=False)
    table.to_csv(OUT / "sweep_trials.csv", index=False)
else:
    table = pd.DataFrame({
        name: {"mean_auc": s["best_val_mean_auc"], "mean_pr_auc": s["best_val_mean_pr_auc"],
               **{f"auc_{k}": s["history"][s["best_epoch"]][f"val_auc_{k}"] for k in LABELS},
               "best_epoch": s["best_epoch"], "minutes": round(s["duration_s"] / 60, 1)}
        for name, s in runs.items()}).T
    table.to_csv(OUT / "ablation_results.csv")
print(table.round(4).to_string())

In [ ]:
# What is kept as output (no comment text: ids, scores, and metrics only).
for p in sorted(OUT.rglob("*")):
    if p.is_file() and "mlruns" not in p.parts:
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT)}")

In [ ]:
# One zip of the small files (ids, scores, metrics, thresholds; no text, no weights).
import zipfile

if MODE == "ablation":

    SMALL = ("summary.json", "metrics.json", "scoring.json", "meta.json", "thresholds.json",
             "thresholds_report.json")
    with zipfile.ZipFile(OUT / "ablation_scored.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted((OUT / "student").rglob("*")):
            rel = p.relative_to(OUT / "student")
            if p.is_file() and "package" not in rel.parts and (
                    p.name in SMALL or rel.parts[1:2] == ("predictions",)):
                z.write(p, rel)
        if (OUT / "ablation_results.csv").exists():
            z.write(OUT / "ablation_results.csv", "ablation_results.csv")
    print("download ablation_scored.zip from the Output tab:",
          round((OUT / "ablation_scored.zip").stat().st_size / 1e6, 1), "MB")
